# 02 · Puntos de quiebre: qué cambia antes y después

**Qué hace.** Corta cada partido del América en **puntos de quiebre** (goles, expulsiones, cambios de sistema, sustituciones tácticas y por lesión, de ambos equipos) y mide qué cambia en los 10 minutos siguientes frente a los 10 anteriores, **en los dos equipos a la vez**. Descuenta lo que cambia solo por el paso del tiempo con **ventanas de control** (otros partidos del mismo entrenador, mismo periodo, mismo marcador, minuto ± 5, sin quiebre). Compara a Jardine con sus predecesores y prueba la **hipótesis de agresividad** (presión en un tramo → qué pasa en el siguiente, y si cambia después del 65').

**Qué necesita.** Pasos 02–05 corridos: `data/processed/` con la columna `tarjeta` (`python scripts/03_procesar.py --rehacer eventos`), `outputs/voronoi/metricas/` (04) y `outputs/panel/` (`python scripts/05_panel.py --procesos 4`).

**Cómo probarlo.** En VS Code, kernel `.venv`, *Run All*. Tarda 1–3 minutos. Las figuras y la tabla de resultados se guardan en `outputs/exploracion/` y `outputs/hallazgos/` (local, no se suben).

**Criterio de hallazgo** (fijado el 12 oct, antes de ver resultados; `contexto/framework.md`): un efecto es hallazgo si (1) su IC 95 % por bootstrap de partidos no cruza 0, (2) mide al menos 0.5 DE de la variación entre partidos de esa métrica y (3) va en la misma dirección en al menos 4 torneos de Jardine.

> IMPOR1: el **Clausura 2026 es el torneo de prueba** y aquí no se usa (`INCLUIR_PRUEBA = False`): los hallazgos alimentan los principios que luego se validan ahí. Con 5 torneos de entrenamiento (AP23–AP25), la condición 3 pide 4 de 5.
>
> IMPOR2: solo agregados. El notebook se sube sin salidas (nbstripout).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from ojo.config import DIR_SALIDAS, params
from ojo.datos import almacen
from ojo.framework import contexto, quiebres
from ojo.framework import partidos as fp
from ojo.lentes import hallazgo, minuto
from ojo.salidas.estilo import color

P = params()
EQUIPO = P["equipo"]["team_id"]
HALLAZGO, AGRESIVIDAD = P["hallazgo"], P["agresividad"]
RNG = np.random.default_rng(P["simulacion"]["semilla"])
INCLUIR_PRUEBA = False  # IMPOR1: el CL26 solo se abre al final, para confirmar
DIR_FIG = DIR_SALIDAS / "exploracion"
DIR_HALLAZGOS = DIR_SALIDAS / "hallazgos"
for d in (DIR_FIG, DIR_HALLAZGOS):
    d.mkdir(parents=True, exist_ok=True)

ENTRENADORES = {d["manager_id"]: d["nombre"].split()[-1] for d in P["predecesores"]}
ENTRENADORES[P["entrenador"]["manager_id"]] = P["entrenador"]["nombre"].split()[-1]
NUEVO, PREVIOS = list(ENTRENADORES.values())[-1], "Predecesores"
METRICAS = [m for lista in minuto.METRICAS.values() for m in lista]
NOMBRES = {
    "control_campo_rival": "Control en campo rival", "altura_sin_balon": "Altura del bloque sin balón",
    "largo_sin_balon": "Largo del bloque sin balón", "ancho_con_balon": "Ancho con balón",
    "compactacion_sin_balon": "Compactación sin balón (yd²/jugador)", "centralizacion": "Centralización de la red",
    "presiones": "Presiones/min", "faltas": "Faltas/min", "contrapresiones": "Contrapresiones/min",
    "xg": "xG/min", "obv": "OBV/min", "field_tilt": "Field tilt",
    "tiros_transicion_contra": "Tiros en transición en contra/min", "contragolpes_contra": "Contragolpes en contra/min",
}
METRICAS

## 1. Partidos

Partidos válidos del América con Solari, Ortiz o Jardine (sin interinos), sin el torneo de prueba. *Predecesores* = Solari + Ortiz.

In [ ]:
partidos = fp.con_torneo(fp.del_equipo(fp.validos(almacen.cargar("partidos")), EQUIPO))
partidos = partidos[partidos["entrenador_id"].isin(ENTRENADORES)].copy()
partidos["prueba"] = fp.es_prueba(partidos)
if not INCLUIR_PRUEBA:
    partidos = partidos[~partidos["prueba"]]
partidos["entrenador"] = partidos["entrenador_id"].map(ENTRENADORES)
partidos["grupo"] = np.where(partidos["entrenador"] == NUEVO, NUEVO, PREVIOS)
TORNEOS = list(partidos.sort_values("fecha")["torneo_corto"].unique())
TORNEOS_NUEVO = [t for t in TORNEOS if t in set(partidos.loc[partidos["grupo"] == NUEVO, "torneo_corto"])]
ids = partidos["match_id"].astype(int).tolist()

panel = minuto.cargar_panel("minuto").query("match_id in @ids")
aristas = minuto.cargar_panel("aristas").query("match_id in @ids")
print(f"Torneos de {NUEVO} usados: {', '.join(TORNEOS_NUEVO)}")
partidos.groupby("entrenador", sort=False).agg(partidos=("match_id", "size"),
                                               torneos=("torneo_corto", lambda s: ", ".join(s.unique())))

## 2. Quiebres y ventanas

Cada quiebre tiene una ventana antes y una después (10 min, sin el minuto del quiebre ni el de al lado). Las ventanas se cortan en el quiebre que cambia el marcador o el número de jugadores más cercano; si una queda en menos de 4 minutos, el quiebre no se usa (`valida`).

In [ ]:
eventos = almacen.cargar("eventos", ids, columnas=["match_id", "indice", "periodo", "minuto", "tipo", "equipo_id",
                                                  "tiro_resultado", "tarjeta", "formacion", "sustitucion_resultado"])
limites = quiebres.limites_periodo(eventos)
q = quiebres.detectar(eventos, EQUIPO).merge(partidos[["match_id", "grupo", "entrenador", "torneo_corto"]], on="match_id")
q = quiebres.ventanas(q, limites)
conteo = q.groupby(["tipo", "grupo"]).agg(quiebres=("valida", "size"), validos=("valida", "sum")).unstack("grupo")
conteo.reindex(quiebres.TIPOS)

## 3. Ventanas de control

Para cada quiebre válido: hasta 20 ventanas de **otros partidos del mismo grupo** (Jardine con Jardine, predecesores con predecesores), en el mismo periodo, con el mismo estado del marcador y minuto ± 5, sin quiebres que corten dentro. Cuánto cambian ahí las métricas es lo que habría cambiado sin el quiebre.

In [ ]:
segmentos = quiebres.segmentos_libres(q, limites).merge(partidos[["match_id", "grupo"]], on="match_id")
controles = quiebres.controles(q, segmentos, por=("grupo",))
controles["id"] = controles["quiebre_id"] + "|c" + controles.groupby("quiebre_id").cumcount().astype(str)
por_quiebre = controles.groupby("quiebre_id").size()
validos = q[q["valida"]]
print(f"Quiebres válidos: {len(validos):,}; con al menos un control: {validos['quiebre_id'].isin(por_quiebre.index).sum():,}; "
      f"controles por quiebre: mediana {por_quiebre.median():.0f}")

## 4. Métricas de cada ventana, a dos bandas

`propio` = América y `rival` = el rival en la misma ventana. Las tasas van por minuto; las métricas de estructura promedian los frames 360 de la ventana.

In [ ]:
def a_ventanas(tabla: pd.DataFrame) -> pd.DataFrame:
    # Ventanas antes y después de cada fila (id, match_id, periodo y límites) en formato largo.
    return pd.concat([pd.DataFrame({"ventana_id": tabla["id"] + "|" + fase, "match_id": tabla["match_id"],
                                    "periodo": tabla["periodo"], "desde": tabla[f"{fase}_desde"],
                                    "hasta": tabla[f"{fase}_hasta"]})
                      for fase in ("antes", "despues")], ignore_index=True)


ventanas = pd.concat([a_ventanas(validos.assign(id=validos["quiebre_id"])), a_ventanas(controles)], ignore_index=True)
medidas = minuto.medir(panel, aristas, ventanas)
medidas[["id", "fase"]] = medidas["ventana_id"].str.rsplit("|", n=1, expand=True)
largo = medidas.melt(id_vars=["id", "fase", "lado"], value_vars=METRICAS, var_name="metrica")
cambio = largo.pivot_table(index=["id", "lado", "metrica"], columns="fase", values="value").reset_index()
cambio["cambio"] = cambio["despues"] - cambio["antes"]
cambio["quiebre_id"] = cambio["id"].str.split("|c", regex=False).str[0]
cambio["es_control"] = cambio["id"].str.contains("|c", regex=False)

base_control = (cambio[cambio["es_control"]].groupby(["quiebre_id", "lado", "metrica"])["cambio"]
                .agg(cambio_control="mean", n_controles="size"))
efectos = (cambio[~cambio["es_control"]].join(base_control, on=["quiebre_id", "lado", "metrica"])
           .merge(q[["quiebre_id", "match_id", "tipo", "grupo", "torneo_corto", "minuto", "estado_antes"]], on="quiebre_id"))
efectos["efecto"] = efectos["cambio"] - efectos["cambio_control"]
print(f"Ventanas medidas: {len(medidas) // 2:,}; efectos (quiebre × lado × métrica): {efectos['efecto'].notna().sum():,}")
efectos.groupby(["grupo", "lado"])["efecto"].count().unstack()

## 5. Variación entre partidos

La condición 2 del criterio compara cada efecto con la desviación estándar de la métrica **entre partidos** (valor del partido completo), por lado.

In [ ]:
minutos_partido = panel[minuto.CLAVES_MINUTO].drop_duplicates().assign(partido=lambda d: d["match_id"])
por_partido = minuto.metricas(minuto.sumar(panel, minutos_partido, "partido"), "partido",
                              minuto.ventanas_red(aristas, minutos_partido, "partido"))
DE = por_partido.groupby("lado")[METRICAS].std().T
DE.round(4)

## 6. Efecto de cada quiebre con Jardine

Una fila por tipo de quiebre, lado y métrica: efecto medio (cambio tras el quiebre menos el de sus controles), IC 95 % por bootstrap de partidos, tamaño en DE entre partidos y en cuántos torneos va en la misma dirección. `pasa` = cumple las tres condiciones; `muestra_chica` = menos de 10 partidos (aviso, no condición).

In [ ]:
def evaluar_grupo(tabla: pd.DataFrame) -> pd.DataFrame:
    # Aplica el criterio de hallazgo a cada (tipo, lado, métrica) de una tabla de efectos.
    filas = []
    for (tipo, lado, metrica), g in tabla.dropna(subset=["efecto"]).groupby(["tipo", "lado", "metrica"]):
        medias = hallazgo.medias_bootstrap(g, "efecto", RNG)
        bajo, alto = hallazgo.intervalo(medias)
        efecto = g.groupby("match_id")["efecto"].sum().sum() / len(g)
        misma = hallazgo.direcciones(g.groupby("torneo_corto")["efecto"].mean(), efecto)
        filas.append({"tipo": tipo, "lado": lado, "metrica": metrica, "quiebres": len(g),
                      "partidos": g["match_id"].nunique(), "efecto": efecto, "ic_bajo": bajo, "ic_alto": alto,
                      "torneos_misma_direccion": misma,
                      "muestra_chica": g["match_id"].nunique() < HALLAZGO["partidos_aviso"],
                      **hallazgo.evaluar(efecto, bajo, alto, DE.loc[metrica, lado], misma)})
    return pd.DataFrame(filas)


jardine = evaluar_grupo(efectos[efectos["grupo"] == NUEVO])
jardine.to_parquet(DIR_HALLAZGOS / "quiebres_jardine.parquet", index=False)
print(f"Pruebas: {len(jardine)} · pasan el IC: {jardine['ic_sin_cero'].sum()} · "
      f"pasan las tres condiciones: {jardine['pasa'].sum()}")
columnas = ["tipo", "lado", "metrica", "quiebres", "partidos", "efecto", "ic_bajo", "ic_alto", "tamano_de",
            "torneos_misma_direccion", "muestra_chica", "pasa"]
jardine.sort_values(["pasa", "ic_sin_cero", "tamano_de"], ascending=[False, False, False],
                    key=lambda s: s.abs() if s.name == "tamano_de" else s)[columnas].head(20).round(3)

### Mapa de efectos

Tamaño del efecto en DE entre partidos (dorado = sube tras el quiebre, azul = baja). Con borde blanco: pasa el criterio; con punto: el IC no cruza 0 pero falla otra condición.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

mapa = LinearSegmentedColormap.from_list("oda", [color("azul"), color("gris-fondo"), color("dorado")])
fig, ejes = plt.subplots(1, 2, figsize=(15, 6.5), sharey=True)
for eje, lado in zip(ejes, ("propio", "rival")):
    sub = jardine[jardine["lado"] == lado]
    tam = sub.pivot(index="tipo", columns="metrica", values="tamano_de").reindex(index=quiebres.TIPOS, columns=METRICAS)
    imagen = eje.imshow(tam.to_numpy(dtype=float), cmap=mapa, vmin=-1, vmax=1, aspect="auto")
    for (i, tipo) in enumerate(tam.index):
        for (j, metrica) in enumerate(tam.columns):
            fila = sub[(sub["tipo"] == tipo) & (sub["metrica"] == metrica)]
            if fila.empty:
                continue
            if bool(fila["pasa"].iloc[0]):
                eje.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, ec="white", lw=2.5))
            elif bool(fila["ic_sin_cero"].iloc[0]):
                eje.plot(j, i, "o", color=color("gris-texto"), ms=3)
    eje.set_xticks(range(len(METRICAS)), [NOMBRES[m] for m in METRICAS], rotation=60, ha="right", fontsize=8)
    eje.set_yticks(range(len(quiebres.TIPOS)), quiebres.TIPOS, fontsize=8)
    eje.set_title(f"{'América' if lado == 'propio' else 'Rival'} · {NUEVO}", fontsize=11)
fig.colorbar(imagen, ax=ejes, shrink=.7, label="Efecto (DE entre partidos)")
fig.savefig(DIR_FIG / "02_mapa_efectos.png", dpi=130, bbox_inches="tight")
plt.show()

## 7. Jardine frente a sus predecesores

Diferencia del efecto de cada quiebre (Jardine − predecesores), con bootstrap de partidos independiente en cada grupo. La condición 3 compara el efecto de Jardine en cada uno de sus torneos con el de los predecesores.

In [ ]:
filas = []
for (tipo, lado, metrica), g in efectos.dropna(subset=["efecto"]).groupby(["tipo", "lado", "metrica"]):
    nuevo, previo = g[g["grupo"] == NUEVO], g[g["grupo"] == PREVIOS]
    if nuevo["match_id"].nunique() < 2 or previo["match_id"].nunique() < 2:
        continue
    medias = hallazgo.medias_bootstrap(nuevo, "efecto", RNG) - hallazgo.medias_bootstrap(previo, "efecto", RNG)
    bajo, alto = hallazgo.intervalo(medias)
    diferencia = nuevo["efecto"].mean() - previo["efecto"].mean()
    por_torneo = nuevo.groupby("torneo_corto")["efecto"].mean() - previo["efecto"].mean()
    misma = hallazgo.direcciones(por_torneo, diferencia)
    filas.append({"tipo": tipo, "lado": lado, "metrica": metrica, "quiebres_nuevo": len(nuevo),
                  "quiebres_previos": len(previo), "efecto_nuevo": nuevo["efecto"].mean(),
                  "efecto_previos": previo["efecto"].mean(), "diferencia": diferencia, "ic_bajo": bajo, "ic_alto": alto,
                  "torneos_misma_direccion": misma,
                  "muestra_chica": min(nuevo["match_id"].nunique(), previo["match_id"].nunique()) < HALLAZGO["partidos_aviso"],
                  **hallazgo.evaluar(diferencia, bajo, alto, DE.loc[metrica, lado], misma)})
contra_previos = pd.DataFrame(filas)
contra_previos.to_parquet(DIR_HALLAZGOS / "quiebres_contra_predecesores.parquet", index=False)
print(f"Pruebas: {len(contra_previos)} · pasan el IC: {contra_previos['ic_sin_cero'].sum()} · "
      f"pasan las tres condiciones: {contra_previos['pasa'].sum()}")
contra_previos.sort_values(["pasa", "ic_sin_cero"], ascending=False).head(15).round(3)

## 8. Cómo se ve un quiebre minuto a minuto

Promedio por minuto alrededor de los goles (minuto 0 = el gol), América y rival, frente a sus ventanas de control. Es la forma del efecto; la prueba formal es la de las secciones 6 y 7.

In [ ]:
RELATIVOS = range(-P["quiebres"]["ventana_min"], P["quiebres"]["ventana_min"] + 1)


def trayectoria(tabla: pd.DataFrame, columna: str) -> pd.DataFrame:
    # Promedio de una columna del panel por minuto relativo y lado (tabla: match_id, periodo, minuto de referencia).
    filas = tabla[["match_id", "periodo", "minuto"]].rename(columns={"minuto": "m0"})
    filas = filas.merge(pd.DataFrame({"relativo": list(RELATIVOS)}), how="cross")
    filas["minuto"] = filas["m0"] + filas["relativo"]
    unidos = filas.merge(panel[[*minuto.CLAVES_MINUTO, "lado", columna]], on=minuto.CLAVES_MINUTO)
    return unidos.groupby(["relativo", "lado"])[columna].mean().unstack("lado")


fig, ejes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for eje, tipo in zip(ejes, ("gol_favor", "gol_contra")):
    sel = validos[(validos["tipo"] == tipo) & (validos["grupo"] == NUEVO)]
    ctrl = controles[controles["quiebre_id"].isin(sel["quiebre_id"])]
    for tabla, estilo, etiqueta in ((sel, "-", "tras el gol"), (ctrl, ":", "control")):
        t = trayectoria(tabla, "presiones").rolling(3, center=True, min_periods=1).mean()
        eje.plot(t.index, t["propio"], estilo, color=color("dorado"), lw=2, label=f"América · {etiqueta}")
        eje.plot(t.index, t["rival"], estilo, color=color("azul"), lw=2, label=f"Rival · {etiqueta}")
    eje.axvspan(-1.5, 1.5, color="0.9")
    eje.set_title(f"{tipo.replace('_', ' ')} ({len(sel)} con {NUEVO})")
    eje.set_xlabel("Minutos desde el gol")
    eje.set_xticks(range(min(RELATIVOS), max(RELATIVOS) + 1, 2))
ejes[0].set_ylabel("Presiones por minuto (media móvil de 3)")
ejes[1].legend(fontsize=8, frameon=False)
fig.savefig(DIR_FIG / "02_presion_alrededor_de_goles.png", dpi=130, bbox_inches="tight")
plt.show()

## 9. Hipótesis de agresividad

Tramos de 5 minutos dentro de cada periodo. ¿La presión del América en campo rival en un tramo se relaciona con lo que pasa en el **siguiente**: su xG, sus faltas, y los contragolpes y tiros en transición del rival? Modelo con efectos fijos por partido y por marcador, la respuesta del tramo actual como control, errores agrupados por partido. `tardío` = tramos desde el 65'. El coeficiente está en unidades de la respuesta por **+1 DE de presión**. Es una relación temporal, no causalidad.

In [ ]:
T, TARDIO = AGRESIVIDAD["tramo_min"], AGRESIVIDAD["tardio_min"]
pm = panel.copy()
inicio, fin = pm["periodo"].map(contexto.INICIO_PERIODO), pm["periodo"].map(contexto.FIN_PERIODO)
pm["k"] = (np.minimum(pm["minuto"], fin) - inicio) // T  # el añadido queda como un tramo propio
claves = ["match_id", "periodo", "k"]
sumas = pm.groupby([*claves, "lado"])[["presiones_campo_rival", "xg", "faltas", "contragolpes", "tiros_transicion"]].sum()
sumas = sumas.unstack("lado", fill_value=0)
n_min = pm.groupby(claves)["minuto"].nunique()
tramos = pd.DataFrame({
    "presion": sumas[("presiones_campo_rival", "propio")] / n_min,
    "xg_favor": sumas[("xg", "propio")] / n_min,
    "faltas": sumas[("faltas", "propio")] / n_min,
    "contragolpes_contra": sumas[("contragolpes", "rival")] / n_min,
    "transicion_contra": sumas[("tiros_transicion", "rival")] / n_min,
    "estado": pm.groupby(claves)["estado"].first(),
    "inicio_tramo": (inicio + pm["k"] * T).groupby([pm[c] for c in claves]).first(),
}).reset_index().merge(partidos[["match_id", "grupo", "torneo_corto"]], on="match_id")
RESPUESTAS = ["xg_favor", "faltas", "contragolpes_contra", "transicion_contra"]
siguiente = tramos.groupby(["match_id", "periodo"])[RESPUESTAS].shift(-1).add_suffix("_sig")
tramos = pd.concat([tramos, siguiente], axis=1).dropna(subset=[f"{r}_sig" for r in RESPUESTAS])
tramos["tardio"] = (tramos["inicio_tramo"] >= TARDIO).astype(int)
tramos["presion_z"] = (tramos["presion"] - tramos["presion"].mean()) / tramos["presion"].std()
DE_PARTIDO = {"xg_favor": DE.loc["xg", "propio"], "faltas": DE.loc["faltas", "propio"],
              "contragolpes_contra": DE.loc["contragolpes_contra", "propio"],
              "transicion_contra": DE.loc["tiros_transicion_contra", "propio"]}


def ajustar(datos: pd.DataFrame, respuesta: str):
    formula = f"{respuesta}_sig ~ presion_z * tardio + {respuesta} + C(estado) + C(match_id)"
    return smf.ols(formula, datos).fit(cov_type="cluster", cov_kwds={"groups": datos["match_id"]})


filas = []
for grupo in (NUEVO, PREVIOS):
    datos = tramos[tramos["grupo"] == grupo]
    for r in RESPUESTAS:
        modelo = ajustar(datos, r)
        for momento, combinacion in (("antes del 65'", "presion_z = 0"), ("desde el 65'", "presion_z + presion_z:tardio = 0")):
            prueba = modelo.t_test(combinacion)
            coef, (bajo, alto) = float(prueba.effect[0]), prueba.conf_int(alpha=1 - P["estadistica"]["nivel_confianza"])[0]
            misma = 0
            if grupo == NUEVO:
                por_torneo = pd.Series({t: float(ajustar(d, r).t_test(combinacion).effect[0])
                                        for t, d in datos.groupby("torneo_corto")})
                misma = hallazgo.direcciones(por_torneo, coef)
            filas.append({"grupo": grupo, "respuesta": r, "momento": momento, "coef_por_DE_presion": coef,
                          "ic_bajo": bajo, "ic_alto": alto, "tramos": int(modelo.nobs), "torneos_misma_direccion": misma,
                          **hallazgo.evaluar(coef, bajo, alto, DE_PARTIDO[r], misma)})
agresividad = pd.DataFrame(filas)
agresividad.loc[agresividad["grupo"] == PREVIOS, "pasa"] = False  # el criterio se evalúa en Jardine
agresividad.to_parquet(DIR_HALLAZGOS / "agresividad.parquet", index=False)
agresividad.round(4)

## 10. Qué pasa y qué no

Resumen de las pruebas de este notebook con el criterio fijado antes de verlas. Con 95 % de confianza, alrededor de 1 de cada 20 pruebas pasaría la condición 1 por azar; las condiciones 2 y 3 están para filtrar esos falsos positivos, pero conviene leer los hallazgos aislados con cautela.

In [ ]:
resumen = pd.DataFrame([
    {"análisis": f"Efecto de quiebres con {NUEVO}", "pruebas": len(jardine), "IC sin cero": jardine["ic_sin_cero"].sum(),
     "pasan": jardine["pasa"].sum()},
    {"análisis": f"{NUEVO} frente a predecesores", "pruebas": len(contra_previos),
     "IC sin cero": contra_previos["ic_sin_cero"].sum(), "pasan": contra_previos["pasa"].sum()},
    {"análisis": "Agresividad (Jardine)", "pruebas": int((agresividad["grupo"] == NUEVO).sum()),
     "IC sin cero": agresividad.loc[agresividad["grupo"] == NUEVO, "ic_sin_cero"].sum(),
     "pasan": agresividad["pasa"].sum()},
])
resumen["por azar (≈5 %)"] = (resumen["pruebas"] * 0.05).round(1)
resumen["pasan con 10+ partidos"] = [int((t["pasa"] & ~t["muestra_chica"]).sum()) for t in (jardine, contra_previos)] + [int(agresividad["pasa"].sum())]

estructura = set(minuto.METRICAS["estructura"])
voronoi_pasa = pd.concat([jardine, contra_previos])
voronoi_pasa = voronoi_pasa[voronoi_pasa["metrica"].isin(estructura) & voronoi_pasa["pasa"]]
print("Voronoi y forma (bloque de estructura):",
      "SIGUE como lente estructural" if len(voronoi_pasa) else "queda como APOYO VISUAL (ninguna métrica pasa)")
resumen

In [ ]:
pasan = pd.concat([
    jardine[jardine["pasa"]].assign(análisis=f"Quiebre con {NUEVO}", valor=lambda d: d["efecto"]),
    contra_previos[contra_previos["pasa"]].assign(análisis=f"{NUEVO} − predecesores", valor=lambda d: d["diferencia"]),
], ignore_index=True)
pasan["métrica"] = pasan["metrica"].map(NOMBRES)
pasan[["análisis", "tipo", "lado", "métrica", "valor", "ic_bajo", "ic_alto", "tamano_de", "torneos_misma_direccion",
       "muestra_chica"]].sort_values(["muestra_chica", "análisis", "tipo"]).round(3)

## Lectura

Corrida del 12 oct (AP21–AP25, sin el CL26). Los números cambian si cambian los datos o `params.yaml`.

1. **El método lee bien el partido.** Las expulsiones a favor (15 con Jardine) mueven todo en la dirección esperada: el América sube su field tilt (+0.35), controla 7.6 puntos más de espacio en campo rival (Voronoi) y presiona menos; el rival hunde su bloque 16.5 yd y se compacta. Cualquier equipo con uno más hace eso: no es un principio de Jardine, es la prueba de que las ventanas y los controles funcionan. Lo mismo vale para buena parte de la sección 6 (efectos del propio juego, no del entrenador).
2. **Lo que distingue a Jardine está en la sección 7** (frente a Solari y Ortiz, mismo quiebre, misma lectura):
    - **Tras anotar** (126 goles con Jardine, 73 con predecesores; 5 de 5 torneos): el América concede menos tiros en transición, sus faltas casi no cambian y el rival contrapresiona menos; con los predecesores las tres subían. Candidato a principio: *ganando, cierra la transición*.
    - **Tras recibir gol** (62 frente a 61): el OBV del América sube más y el rival contrapresiona y comete menos faltas. Candidato: *reacción al gol en contra*.
    - **Sustituciones tácticas propias** (264 frente a 206; 5 de 5 torneos): el control en campo rival sube con Jardine y baja con los predecesores (diferencia de 1.4 puntos, 0.76 DE). Es el hallazgo de Voronoi con más muestra: *sus cambios ganan espacio arriba*.
    - **Cambios de sistema propios** (44 frente a 13): con Jardine van seguidos de más faltas, menos xG y más tiros en transición en contra; con los predecesores, al revés. La muestra de los predecesores es corta y el cambio de sistema suele responder a algo que ya pasaba (selección), así que es una pista, no un principio.
3. **Lo que no pasa.** Por sí solas, las sustituciones tácticas de Jardine no cambian de forma medible el juego del América en 10 minutos (0 de 28 pruebas). En la hipótesis de agresividad ninguna respuesta pasa: con Jardine, más presión antes del 65' va con algo más de tiros en transición en contra en el tramo siguiente (IC sin cero), pero el tamaño es de 0.26 DE (< 0.5), y después del 65' no hay señal.
4. **Voronoi.** Por la regla fijada, **sigue como lente estructural**: control en campo rival y forma del bloque pasan tras varios quiebres. El más sólido es el de las sustituciones tácticas frente a los predecesores; los de expulsiones son mecánicos.
5. **Cautelas.** Fueron 566 pruebas: unas 28 pasarían la condición 1 por azar y la pasaron 80. La condición 2 filtra poco en los quiebres (OBS1 de `framework.md`), así que pesan la 1 y la 3. Todo es comparación emparejada (diferencia en diferencias), no causalidad estricta: sustituir o cambiar de sistema es una decisión que responde a lo que pasaba en el partido. Estos candidatos se confirman al final en el CL26.